### Implement a solution to handle Slowly Changing Dimension for both Initial and Incremental Run in one Notebook

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from delta.tables import DeltaTable

-----INPUT DATA ---DIM TABLE

# Slowly Changing Dimension - Initial and Incremental

In [0]:
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/FileStore/rawcsv")

In [0]:
df = df.select("p_id","p_name","p_category").filter(col("p_id").isNotNull())

In [0]:
df.display()

First try initial run =1 and after that below 

In [0]:
initial_run = 0

In [0]:
if (initial_run == 0):
    delta_table = DeltaTable.forPath(spark,"/FileStore/rawcsvsink2")

    delta_table.alias("trg").merge(df.alias("src"), "trg.p_id = src.p_id")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    df.write.format("delta")\
            .mode("append")\
            .option("path","/FileStore/rawcsvsink2")\
            .saveAsTable("productsDim")

In [0]:
%sql
SELECT * FROM productsdim

**RELATED TO LOAD **TYPES****

NOTE
Interview Answer
If asked:
"What is the difference between Batch Load, Incremental Load, Full Refresh, and Incremental Refresh?"
You can answer:
Batch Load refers to when data is processed (hourly, daily, weekly).
Incremental Load refers to loading only new or changed records.
Full Refresh replaces the entire target dataset every run.
Incremental Refresh updates only the changed portion of the target dataset without reloading everything.
Easy Formula
Load Type Refresh Type
------------------------------------------------
Initial Load → Full Refresh
Daily Full Load → Full Refresh
Daily Changed Data → Incremental Refresh

Batch = WHEN data runs
Refresh = HOW target is updated
This last line is how most Fabric, Databricks, ADF, and Synapse interviewers expect you to explain it:
Batch tells WHEN the pipeline runs. Refresh tells HOW the target data is updated.



**scd explanation**

**What happens during the Load?**
From the screenshot, there are 2 scenarios:
1. Initial Load (initial_run = 1)
df.write.format("delta") \
.mode("append") \
.option("path","/Filestore/rawcsvsink2") \
.saveAsTable("productsDim")
Input (Source)
p_id	Product
1	Laptop
2	Mobile
3	Tablet
Target
Empty
Result
p_id	Product
1	Laptop
2	Mobile
3	Tablet
Explanation:
•	Creates the Delta table.
•	Loads all records from source.
•	No comparison with existing data.
•	This is a Full/Initial Load.
________________________________________
2. Incremental Load (initial_run = 0)
Existing Target
p_id	Product
1	Laptop
2	Mobile
3	Tablet
New Source File
p_id	Product
1	Gaming Laptop
2	Mobile
4	Smart Watch
MERGE Logic
delta_table.alias("trg").merge(
df.alias("src"),
"trg.p_id = src.p_id"
)\
.whenMatchedUpdateAll()\
.whenNotMatchedInsertAll()\
.execute()
Processing
Record 1
Target : Laptop
Source : Gaming Laptop
Match found →
Update
Record 2
Target : Mobile
Source : Mobile
Match found →
No visible change
Record 4
Does not exist in target
Insert new row.
________________________________________
Final Target Table
p_id	Product
1	Gaming Laptop
2	Mobile
3	Tablet
4	Smart Watch
________________________________________
Load Type Classification
First Run
initial_run = 1
✅ Batch Load
✅ Full Load
✅ Full Refresh
________________________________________
Later Runs
initial_run = 0
✅ Batch Load
✅ Incremental Load
✅ Incremental Refresh
✅ Upsert (Update + Insert)
________________________________________
Interview Explanation
During the first run, the notebook performs a full load and creates the Delta table by loading all source records. In subsequent runs, it uses a Delta MERGE operation based on the business key (p_id). Existing records are updated using whenMatchedUpdateAll(), and new records are inserted using whenNotMatchedInsertAll(). This implements an incremental upsert pattern similar to SCD Type 1.


**How Do We Identify Initial Load in Real Projects?**

How Do We Identify Initial Load in Real Projects?
We do NOT hardcode:
initial_run = 1
Instead, we determine it dynamically.
________________________________________
Scenario 1: Table Existence + Count Check
Handles all cases:
•	Table does not exist
•	Table exists but empty
•	Table exists and contains data
PySpark Logic
if not spark.catalog.tableExists("ProductsDim"):
print("Initial Load")
else:
count = spark.table("ProductsDim").count()
if count == 0:
print("Initial Load")
else:
print("Incremental Load")
________________________________________
Case 1
ProductsDim does not exist
Result
Initial Load
________________________________________
Case 2
ProductsDim exists
Rows = 0
Result
Initial Load
________________________________________
Case 3
ProductsDim exists
Rows = 10000
Result
Incremental Load
________________________________________

Scenario 2: Pipeline Parameter
Pipeline passes:
LoadType = Initial
or
LoadType = Incremental

________________________________________
PySpark
load_type = dbutils.widgets.get("LoadType")
if load_type == "Initial":
print("Initial Load")
else:
print("Incremental Load")
Used For
Historical Load
Migration
Backfill
Manual Execution

________________________________________
Scenario 3: Control Table
Enterprise Standard Approach.
________________________________________
Control Table
TableName InitialFlag
ProductsDim N
________________________________________
Read Flag
flag = spark.sql("""
SELECT InitialFlag
FROM ControlTable
WHERE TableName='ProductsDim'
""").collect()[0][0]
________________________________________
Logic
if flag == "N":
print("Initial Load")
else:
print("Incremental Load")
________________________________________
Meaning
N = Initial Load Not Completed
Y = Initial Load Completed
________________________________________
After Successful Load
UPDATE ControlTable
SET InitialFlag='Y'
WHERE TableName='ProductsDim';
________________________________________
Next Run
InitialFlag = Y
Result
Incremental Load
________________________________________
Scenario 4: Watermark (Most Common Enterprise Approach)
Most commonly used in:
•	Microsoft Fabric
•	Azure Data Factory
•	Azure Databricks
•	Synapse
•	Snowflake
________________________________________
What is a Watermark?
Watermark = Last Successful Load Timestamp
________________________________________
Control Table
TableName LastLoadDate

ProductsDim NULL
________________________________________
Read Watermark
watermark = spark.sql("""
SELECT LastLoadDate
FROM ControlTable
WHERE TableName='ProductsDim'
""").collect()[0][0]
________________________________________
Initial Run
LastLoadDate = NULL
Logic
if watermark is None:
Result
Initial Load
________________________________________
Incremental Run
LastLoadDate = 2026-08-09 10:00:00
Result
Incremental Load
________________________________________
Incremental Query
source_df = spark.sql(f"""
SELECT *
FROM ProductSource
WHERE ModifiedDate > '{watermark}'
""")
________________________________________
Most Important Doubt
Does this compare Source and Target?
WHERE ModifiedDate > LastLoadDate
Answer
❌ NO
It compares:
Source Table
VS
Control Table
________________________________________
Example
Source
ProductID ModifiedDate

1 09:00
2 11:00
3 12:00
Watermark
10:00
Output
ProductID

2
3
Only changed source records are selected.
________________________________________
Easy Formula
Watermark
=
Source Table
VS
Control Table
Example:
ModifiedDate > LastLoadDate
________________________________________
Then Where Does Source vs Target Happen?
deltaTable.alias("trg").merge(
source_df.alias("src"),
"trg.ProductID = src.ProductID"
)
________________________________________
MERGE Comparison
Source ProductID
VS
Target ProductID
________________________________________
Example
Target
1 Laptop
2 Old Mobile
Source
2 Mobile
3 Tablet
________________________________________
ProductID = 2
Match Found
Action:
UPDATE
________________________________________
ProductID = 3
No Match Found
Action:
INSERT
________________________________________
Easy Formula
MERGE
=
Source Table
VS
Target Table
________________________________________
Complete Enterprise PySpark Logic
from delta.tables import DeltaTable

watermark = spark.sql("""
SELECT LastLoadDate
FROM ControlTable
WHERE TableName='ProductsDim'
""").collect()[0][0]

if watermark is None:

# Initial Load

source_df = spark.sql("""
SELECT *
FROM ProductSource
""")

source_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("ProductsDim")

else:

# Incremental Load

source_df = spark.sql(f"""
SELECT *
FROM ProductSource
WHERE ModifiedDate > '{watermark}'
""")

deltaTable = DeltaTable.forName(
spark,
"ProductsDim"
)

deltaTable.alias("trg").merge(
source_df.alias("src"),
"trg.ProductID = src.ProductID"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()
________________________________________
Watermark Update Logic
After successful load:
from pyspark.sql.functions import max

new_watermark = source_df.agg(
max("ModifiedDate")
).collect()[0][0]
Update Control Table:
UPDATE ControlTable
SET LastLoadDate = 'new_watermark'
WHERE TableName = 'ProductsDim';
________________________________________
Interview Cheat Sheet
Initial Load
Table Not Exists
OR
Table Exists + Count = 0
OR
InitialFlag = N
OR
LastLoadDate = NULL
Result
Initial Load
________________________________________
Incremental Load
Table Exists + Count > 0
OR
InitialFlag = Y
OR
LastLoadDate Available
Result
Incremental Load
________________________________________
Golden Interview Answer
In real-world Fabric, Databricks, and ADF projects, we don't hardcode initial_run=1. We determine the load type dynamically using table existence, row count, pipeline parameters, control tables, or watermarks. The most common enterprise approach is a Control Table with a Watermark (LastLoadDate). If the watermark is NULL, an Initial Load is performed. Otherwise, records with ModifiedDate > LastLoadDate are extracted from the source and merged into the target using a business key. Watermark compares Source vs Control Table, while MERGE compares Source vs Target Table.
